# Play CartPole

Choose an environment with the buttons, preview an `rgb_array` frame inline, then run the last cell to play in a pygame window.

In [ ]:
import ipywidgets as widgets
import numpy as np
from IPython.display import display
from PIL import Image
from masa.common.notebook_play import make_reset_env, sync_selected_env

VALID_ENV_NAMES = ("DiscreteCartpole", "ContinuousCartpole")

def make_env(env_name, **kwargs):
    if env_name == "DiscreteCartpole":
        from masa.envs.discrete.cartpole import DiscreteCartpole

        return DiscreteCartpole(**kwargs)
    if env_name == "ContinuousCartpole":
        from masa.envs.continuous.cartpole import ContinuousCartpole

        return ContinuousCartpole(**kwargs)
    raise ValueError(f"env_name must be one of {VALID_ENV_NAMES!r}")

ENV_SELECTOR = widgets.ToggleButtons(
    options=VALID_ENV_NAMES,
    value="DiscreteCartpole",
    description="Env",
)
display(ENV_SELECTOR)

ENV_NAME = ENV_SELECTOR.value
SEED = 0


In [ ]:
ENV_NAME = ENV_SELECTOR.value
preview_env = make_env(ENV_NAME, render_mode="rgb_array", render_window_size=512)
obs, info = preview_env.reset(seed=SEED)
print("reset:", {"obs": obs, "info": info})
display(Image.fromarray(preview_env.render()))
preview_env.close()


In [ ]:
def action_from_pressed(env_name, pressed, pygame):
    if env_name == "DiscreteCartpole":
        if pressed[pygame.K_LEFT] or pressed[pygame.K_a]:
            return 0
        if pressed[pygame.K_RIGHT] or pressed[pygame.K_d] or pressed[pygame.K_SPACE]:
            return 1
        return None

    if pressed[pygame.K_LEFT] or pressed[pygame.K_a]:
        return np.array([-1.0], dtype=np.float32)
    if pressed[pygame.K_RIGHT] or pressed[pygame.K_d] or pressed[pygame.K_SPACE]:
        return np.array([1.0], dtype=np.float32)
    if pressed[pygame.K_DOWN] or pressed[pygame.K_s]:
        return np.array([0.0], dtype=np.float32)
    return None


def _print_reset(obs, info):
    print("reset:", {"obs": obs, "info": info})


def play_env(env_name=None, seed=SEED):
    import pygame

    follow_selector = env_name is None
    if follow_selector:
        env_name = ENV_SELECTOR.value
    env, obs, info = make_reset_env(make_env, env_name, seed=seed, render_mode="human", render_window_size=512)
    finished = False
    print("Controls: Left/A negative force, Right/D/Space positive force, Down/S neutral for continuous, R resets, Q or Escape quits.")
    _print_reset(obs, info)

    try:
        running = True
        while running and not env.human_window_closed:
            if follow_selector:
                env, env_name, obs, info, switched = sync_selected_env(
                    env,
                    env_name,
                    ENV_SELECTOR,
                    make_env,
                    seed=seed,
                    render_window_size=512,
                    pygame=pygame,
                )
                if switched:
                    finished = False
                    print("switched:", env_name)
                    _print_reset(obs, info)

            for event in pygame.event.get():
                if not env.handle_pygame_event(event):
                    running = False
                    break
                if event.type != pygame.KEYDOWN:
                    continue
                if event.key in (pygame.K_q, pygame.K_ESCAPE):
                    running = False
                    break
                if event.key == pygame.K_r:
                    obs, info = env.reset(seed=seed)
                    finished = False
                    _print_reset(obs, info)
                    continue

            if not running:
                break

            action = None if finished else action_from_pressed(env_name, pygame.key.get_pressed(), pygame)

            if action is None:
                env.render()
                continue

            obs, reward, terminated, truncated, info = env.step(action)
            finished = terminated or truncated
            print({
                "obs": obs,
                "reward": reward,
                "terminated": terminated,
                "truncated": truncated,
                "info": info,
            })
    finally:
        env.close()

play_env(seed=SEED)